# 07 — MaxViT-512 Downstream

One directly runnable notebook represents one architecture. Change only CONDITION and SEED to cover its 12 protocol jobs.

## 1. Experiment configuration

In [ ]:
from pathlib import Path
import json
import sys
ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'configs').is_dir())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'notebooks/utility'))
from notebooks.utility.downstream_experiment import (
    configure_environment, construct_dataset, experiment_configuration, load_model,
    resume_status, run_validation, saved_artifacts, train, training_budget,
)
ARCHITECTURE = 'maxvit512'
CONDITION = 'real_only'
SEED = 17
GPU = 0
RESUME = True
RUN_TRAINING = False
RUN_VALIDATION = False
configuration = experiment_configuration(ROOT, ARCHITECTURE, CONDITION, SEED, gpu=GPU, resume=RESUME)
configuration['root'] = str(ROOT)
configuration

## 2. Environment and GPU

In [ ]:
environment = configure_environment(configuration)
environment

## 3. Selected generators

In [ ]:
from notebooks.utility.downstream_protocol import load_selected_generators
selected_generators = load_selected_generators(ROOT, required=False)
selected_generators or 'Not required for real-only conditions; run notebook 06 before a synthetic condition.'

## 4. Dataset construction

In [ ]:
dataset = construct_dataset(ROOT, configuration)
len(dataset['train_rows']), len(dataset['validation_rows'])

## 5. Dataset audit

In [ ]:
dataset['audit']  # fails before training if train/validation patients overlap

## 6. Model loading

In [ ]:
model_bundle = None
if RUN_TRAINING:
    model_bundle = load_model(configuration)
model_bundle if model_bundle is not None else 'Deferred until RUN_TRAINING=True; no weights downloaded during protocol review.'

## 7. Trainable parameters

In [ ]:
trainable_parameters = None
if model_bundle is not None:
    _, model = model_bundle
    trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
trainable_parameters if trainable_parameters is not None else 'Not yet instantiated'

## 8. Training configuration

In [ ]:
training_budget(configuration)

## 9. Resume status

In [ ]:
resume_status(ROOT, configuration)

## 10. Training

In [ ]:
training_result = None
if RUN_TRAINING:
    training_result = train(ROOT, configuration, dataset)
else:
    print('Training disabled. Review the audit, then set RUN_TRAINING=True for the selected CONDITION × SEED.')

## 11. Training curves

In [ ]:
history_path = Path(configuration['results_dir']) / 'training_history.csv'
print(history_path if history_path.is_file() else 'Not yet evaluated')

## 12. Best checkpoint

In [ ]:
checkpoint = training_result['checkpoint'] if training_result else None
{'checkpoint': checkpoint, 'criterion': configuration['policy']['checkpoint_criterion'],
 'tie_policy': 'lower validation loss, then earlier epoch'}

## 13. Validation inference

In [ ]:
validation_result = None
if RUN_VALIDATION:
    if checkpoint is None: raise RuntimeError('A validation-selected checkpoint is required')
    validation_result = run_validation(ROOT, configuration, dataset, checkpoint)
validation_result or 'Not yet evaluated'

## 14. Validation metrics

In [ ]:
validation_result['metrics'] if validation_result else 'Not yet evaluated'

## 15. Calibration

In [ ]:
calibration = validation_result['metrics'].get('ece') if validation_result else 'Not yet evaluated'
calibration

## 16. Error analysis

In [ ]:
error_groups = ['false positives', 'false negatives', 'largest calibrated errors']
error_groups

## 17. Interpretability

In [ ]:
interpretability_policy = configuration['policy']['interpretability']
interpretability_policy

## 18. Saved artifacts

In [ ]:
saved_artifacts(ROOT, configuration)